# Manual Implementation of Linear and Quadratic Discriminant Analysis

[Use numpy for all questions (`sklearn` is only allowed to *check* your results)]

In [1]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

## A generative classifier that looks at the whole cloud

Like naive Bayes, these are **generative** classifiers:
they describe what each class looks like, and Bayes' rule turns that description into a prediction.

$$ P(y = k \mid x) \;=\; \frac{P(y = k) \; P(x \mid y = k)}{P(x)} $$

The naive model refused to look at more than one feature at a time:
it declared the features independent given the class, so that $P(x \mid y = k)$ was a product of $p$ one-dimensional densities.
We now drop that assumption and model the class as one $p$-dimensional gaussian cloud,

$$ P(x \mid y = k) \;=\; \mathcal{N}(x \mid \mu_k, \Sigma_k) \;=\; \frac{1}{(2\pi)^{p/2} \, |\Sigma_k|^{1/2}} \exp\Big( -\tfrac{1}{2} (x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k) \Big) $$

- $\mu_k$ is the centre of the class, $p$ numbers, exactly what naive Bayes already had
- $\Sigma_k$ is its covariance matrix, $p(p+1)/2$ numbers: the diagonal holds the per-feature variances naive Bayes already had, and the off-diagonal entries are the new information, they say that two features move together
- fitting is still nothing but averaging: a mean and a covariance per class, no gradient, no iteration

One question is left open, what to do with the $K$ covariance matrices, and the two answers are the two halves of this notebook:

- **LDA** (linear discriminant analysis) gives every class the **same** $\Sigma$, and its boundary comes out a straight line,
- **QDA** (quadratic discriminant analysis) gives every class **its own** $\Sigma_k$, and its boundary comes out a conic.

We build LDA first, then let each class have its own matrix and watch what that does to the boundary.

Two plotting functions are provided:

In [2]:
def plot_data(X, y):
    plt.plot(X[y == 1, 0], X[y == 1, 1], 'o', color='crimson', ms=4, label='class 1')
    plt.plot(X[y == 0, 0], X[y == 0, 1], 's', color='royalblue', ms=4, label='class 0')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')

In [3]:
def plot_regions(predict, X, y, res=200, pad=1.0, title=None):
    g1 = np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, res)
    g2 = np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, res)
    G1, G2 = np.meshgrid(g1, g2)
    Z = np.asarray(predict(np.c_[G1.ravel(), G2.ravel()])).reshape(G1.shape)
    plt.contourf(G1, G2, Z, levels=[-0.5, 0.5, 1.5],
                 colors=['royalblue', 'crimson'], alpha=0.15)
    plot_data(X, y)
    if title:
        plt.title(title)

A covariance matrix is easier to *build* than to read, so start from the picture:
an ellipse of half-axes $s_x$ and $s_y$, turned by an angle $\theta$.
Stretching along the axes is a diagonal matrix, turning is a rotation, and the covariance of the rotated cloud is

$$ \Sigma \;=\; R(\theta) \begin{pmatrix} s_x^2 & 0 \\ 0 & s_y^2 \end{pmatrix} R(\theta)^T
\qquad \text{with} \qquad R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} $$

Write `cov(sx, sy, theta)` returning that $2 \times 2$ matrix.

*At $\theta = 0$ the matrix is diagonal and we are back in the world naive Bayes believes in: the off-diagonal entry is exactly the part of the cloud that a product of two one-dimensional densities cannot express.*

```python
cov(1.0, 1.0, 0.7)       # -> the identity: a round cloud has no orientation
cov(2.0, 0.5, 0.0)       # -> [[4, 0], [0, 0.25]], axis aligned
cov(2.0, 0.5, np.pi/4)   # -> [[2.125, 1.875], [1.875, 2.125]]
```

Now build the dataset. Write a function `blobs(n_per)` returning `X` $(2 n_{per} \times 2)$ and `y`:

- class $0$ is centred on $(-1, 0)$, with spreads $(2, 0.5)$, turned by $+45°$,
- class $1$ is centred on $(1, 0.5)$, with spreads $(2, 0.5)$, turned by $-45°$.

Use `np.random.multivariate_normal(mean, cov, n_per)`, and your `cov` function for the two covariances.
Labels are $y_i \in \{0, 1\}$ (integers, they will be used as indices).

Generate a **training set** of $300$ points ($150$ per class) and a **test set** of $1000$ points ($500$ per class), and plot the training set.

*The two clouds are equally elongated and equally wide, they only lean the other way. Look at the plot and convince yourself that no straight line can use that difference.*

### The density of a cloud that leans

Both models need the building block, the log of the multivariate gaussian density:

$$ \log \mathcal{N}(x \mid \mu, \Sigma) \;=\; -\frac{p}{2} \log (2 \pi) \;-\; \frac{1}{2} \log |\Sigma| \;-\; \frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu) $$

Write it as `log_gaussian(X, mu, Sigma)`, taking an $(n \times p)$ table and returning $n$ numbers, one per row.

Three pieces of advice, all three of which matter again in the last section:

- Do **not** write `np.linalg.inv`. To apply $\Sigma^{-1}$ to the $n$ centred rows at once, solve the system instead: with `D = X - mu`, `np.linalg.solve(Sigma, D.T).T` is $(n \times p)$ and holds $\Sigma^{-1} d_i$ on every row. It is faster and numerically better behaved.
- The quadratic form needs no loop either: it is the row-wise dot product of `D` with that array, so `(D * np.linalg.solve(Sigma, D.T).T).sum(axis=1)` gives the $n$ numbers you want (`np.einsum('ij,ij->i', ...)` does the same).
- For $\log|\Sigma|$, use `np.linalg.slogdet`, which returns the pair (sign, $\log|\det|$). A determinant underflows to $0$ in a few dozen dimensions, its logarithm does not.

*The last term is the squared **Mahalanobis** distance from $x$ to $\mu$: the ordinary distance, measured in units of the cloud's own spread in each direction. A point three units away along the long axis of the ellipse is closer, in that metric, than a point one unit away across it.*

```python
X_mini = np.array([[0., 0.], [1., 0.], [1., 2.]])

log_gaussian(X_mini, np.array([0., 0.]), np.eye(2))                 # -> -1.8379, -2.3379, -4.3379
log_gaussian(X_mini, np.array([1., 0.]), cov(1.5, 0.6, np.pi/3))    # -> -2.8297, -1.7325, -3.7881
```

-----

## LDA: one shape for every class

**Every class is the same cloud, moved somewhere else.**

That is the whole assumption: the classes may sit anywhere, but they are all stretched and turned the same way, so there is a single $\Sigma$ to estimate instead of $K$ of them.

Fitting it is as easy as it sounds. Centre each class on its own mean, pile all the centred rows together, and take the covariance of the pile:

$$ \hat\Sigma \;=\; \frac{1}{n - K} \sum_{k=1}^{K} \; \sum_{i \,:\, y_i = k} (x_i - \mu_k)(x_i - \mu_k)^T $$

This is the **pooled** (or **within-class**) covariance: every row contributes, but each is measured from the centre of *its own* class, so the distance between the classes never enters it.
The divisor is $n - K$ because $K$ means were estimated from the same rows.

Write `fit_lda(X, y)` returning the four things the model is made of:

- `classes`, the distinct labels, in the order given by `np.unique`,
- `priors`, of shape $(K,)$: the proportion of training rows in each class,
- `means`, of shape $(K \times p)$: the mean of every feature within every class,
- `covs`, of shape $(K \times p \times p)$: the pooled matrix above, **repeated $K$ times**.

Repeating it is wasteful, but it buys something: every function written from now on takes one covariance per class, so QDA will drop straight into all of them without a single change.

One loop over the classes is enough: build the $(n \times p)$ array `D` of class-centred rows inside it, and `D.T @ D / (n - K)` is the sum of outer products above.

```python
X_mini = np.array([[0., 0.], [2., 1.], [1., 2.], [1., 1.],
                   [4., 3.], [5., 5.], [3., 4.], [6., 4.]])
y_mini = np.array([0, 0, 0, 0, 1, 1, 1, 1])
model_mini = fit_lda(X_mini, y_mini)

model_mini[1]      # priors -> [0.5, 0.5]
model_mini[2]      # means  -> [[1. , 1. ], [4.5, 4. ]]
model_mini[3][0]   # pooled -> [[1.1667, 0.3333], [0.3333, 0.6667]]
```

Fit it on the training set and print the priors, the means and the pooled covariance.
Compare it with the two matrices you used to generate the data: the two clouds lean opposite ways, so what does averaging them give?

The numerator of Bayes' rule, in log space, is now a sum of two terms instead of $p + 1$:

$$ L_k(x) \;=\; \log \big( P(y = k) \, P(x \mid y = k) \big) \;=\; \log P(y = k) \;+\; \log \mathcal{N}(x \mid \mu_k, \Sigma_k) $$

Write `log_joint(X, model)` returning the $(n \times K)$ array holding that quantity for every row and every class.
One loop over the classes, and inside it a single call to `log_gaussian`.

```python
log_joint(X_mini, model_mini)[:4]      # -> [[ -3.2033, -17.4533],
                                       #     [ -2.8283,  -9.5783],
                                       #     [ -3.2033,  -8.4533],
                                       #     [ -2.3283, -11.0783]]
```

Turn the scores into probabilities.
The missing denominator is $P(x) = \sum_k P(y = k) P(x \mid y = k)$, so

Each term there, numerator and summand alike, is exactly what `log_joint` already computed, read back out of log space:

$$ P(y = k) \, P(x \mid y = k) \;=\; e^{L_k(x)} \qquad \text{with} \qquad L_k(x) \;=\; \log P(y = k) \;+\; \sum_{j=1}^{p} \log \mathcal{N}(x_j \mid \mu_{kj}, \sigma^2_{kj}) $$

Substituting it above and below the bar,

$$ P(y = k \mid x) \;=\; \frac{P(y = k) \, P(x \mid y = k)}{\sum_{k'} P(y = k') \, P(x \mid y = k')} $$

leaves the same quantity on both sides of the fraction, so

$$ P(y = k \mid x) \;=\; \frac{e^{L_k}}{\sum_{k'} e^{L_{k'}}} \qquad \text{where } L_k \text{ is the } k\text{-th column of } \texttt{log\_joint} $$

which is a softmax over the columns.

Print the training and the test accuracy, and plot the decision regions with `plot_regions`.

-----

## QDA: one shape per class

**Give every class its own covariance matrix, and nothing else changes.**

The assumption LDA made is a strong one, and our dataset violates it on purpose: the two clouds lean opposite ways, and pooling them produced a matrix that describes neither.
So estimate them separately, $\hat\Sigma_k$ from the rows of class $k$ alone.

Write `fit_qda(X, y)`, returning the same four-element model as `fit_lda`, with `covs[k]` now the covariance of class $k$.
`np.cov(X[y == c], rowvar=False)` does it in one call: careful with `rowvar=False`, since `np.cov` assumes one variable per *row* by default, and our rows are observations.

*Note that `np.cov` divides by $n_k - 1$, unlike `np.var` in the naive Bayes notebook. That is also what `sklearn` does for QDA, so keep it.*

```python
model_mini = fit_qda(X_mini, y_mini)

model_mini[3]      # -> [[[0.6667, 0.3333], [0.3333, 0.6667]],
                   #     [[1.6667, 0.3333], [0.3333, 0.6667]]]
log_joint(X_mini, model_mini)[:3]      # -> [[-2.9817, -16.6144], [-2.9817, -9.6144], [-2.9817, -7.6144]]
```

Fit it on the training set and print the two covariance matrices. Do they match the ones you used to generate the data?

Then check that nothing was lost: the pooled matrix of `fit_lda` is exactly the weighted average of these two,

$$ \hat\Sigma \;=\; \frac{1}{n-K} \sum_k (n_k - 1) \, \hat\Sigma_k $$

since both sides are the same sum of outer products, grouped differently.

`log_joint`, `predict_proba` and `predict` take the QDA model without a change, since it has the same shape as the LDA one.

Print the training and the test accuracy, and plot the decision regions of LDA and of QDA side by side (`plt.subplot(1, 2, ...)`).

-----

**Summary:**

| model | covariance | boundary | parameters | fails when |
|---|---|---|---|---|
| gaussian naive Bayes | diagonal, per class | axis aligned conic | $2Kp$ | features are correlated |
| LDA | full, shared | a line | $Kp + \frac{p(p+1)}{2}$ | the classes have different shapes |
| QDA | full, per class | a conic | $Kp + K\frac{p(p+1)}{2}$ | $n_k$ is not much larger than $p$ |
